# Random Forest - Sweeps

This notebook demonstrates how to use the Random Forest pipeline for predicting match outcomes between two teams, following the same structure as the XGBoost model.

## Setup and Imports

In [1]:
import sys

sys.path.append("..")

import pandas as pd
import wandb
import dotenv

from src.api.run import sweep_randomforest
from src.api.sweep import wandb_sweep

c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because

In [2]:
dotenv.load_dotenv()
wandb.login()

wandb: Currently logged in as: v8-luky (aicomp-mmlm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Experiments

In [3]:
max_runs = 100
sweep_config = {
    "name": "Random Forest",
    "method": "bayes",
    "metric": {"name": "valid_brier", "goal": "minimize"},
    "parameters": {
        "randomforest_config": {
            "parameters": {
                "n_estimators": {"distribution": "int_uniform", "min": 50, "max": 600},
                "max_depth": {"distribution": "int_uniform", "min": 5, "max": 50},
                "min_samples_split": {"distribution": "int_uniform", "min": 2, "max": 10},
                "min_samples_leaf": {"distribution": "int_uniform", "min": 1, "max": 5},
                "max_features": {"values": ["sqrt", "log2", None, 0.2, 0.5, 0.8]},
                "bootstrap": {"value": True},
                "max_samples": {"distribution": "uniform", "min": 0.2, "max": 0.8},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2024},
                "start_season": {"value": 2003},
                "num_features": {"distribution": "int_uniform", "min": 4, "max": 100},
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_randomforest, run_count=max_runs, project="random-forest")

## Submission from Best Model

In [ ]:
from src.dataloaders.simple import SeasonAverageDataLoader
from src.experiments import DefaultTracker
from src.models.randomforest import RandomForestRegressorModel, RandomForestHyperparamConfig
from src.experiments.config import RunConfig
from src.submissions import generate_matchups, create_submission

In [7]:
run = wandb.Api().run("bc45e7bx")
config = run.config
config

{'run_config': {'num_features': 4, 'start_season': 2003, 'valid_season': 2024},
 'randomforest_config': {'bootstrap': True,
  'max_depth': 6,
  'max_samples': 0.6904031179533447,
  'max_features': 'log2',
  'n_estimators': 261,
  'min_samples_leaf': 2,
  'min_samples_split': 6}}

In [ ]:
run_config = RunConfig(**config.get("run_config", {}))
dataloader = SeasonAverageDataLoader(run_config.num_features)
run_config

RandomForestRunConfig(num_features=4, valid_season=2024, start_season=2003, data_loader='season_average')

In [18]:
hyperparameters = RandomForestHyperparamConfig(**config.get("randomforest_config", {}))
hyperparameters

RandomForestHyperparamConfig(n_estimators=261, max_depth=6, min_samples_split=6, min_samples_leaf=2, max_features='log2', bootstrap=True, max_samples=0.6904031179533447, min_impurity_decrease=0.0, max_leaf_nodes=None, min_weight_fraction_leaf=0.0, random_state=42, n_jobs=-1, verbose=0, warm_start=False, ccp_alpha=0.0)

In [19]:
model = RandomForestRegressorModel(dataloader, hyperparameters, None, DefaultTracker({}))

In [20]:
season = 2025
create_submission(season=season, model=model, filename=f"random_forest_submission_{season}.csv", fit=True)

metrics: {'train_brier': np.float64(0.15205547796147378)}, step: None


WindowsPath('C:/Users/kybur/Repos/HSLU/aicomp/code/submissions/random_forest_submission_2025.csv')